# Progressive Reasoning Fine-Tuning: OOM-Safe LoRA Continuation

This Kaggle-oriented notebook continues training an existing Nemotron LoRA adapter with a controlled auxiliary-data mixture. Its default configuration uses **10% Replay Math**, a low learning rate, dynamic micro-batching, and MoE-aware LoRA weight tying.

The notebook is the continued-training and ablation branch of the broader progressive reasoning solution. OpenMath support remains available behind configuration switches, while the checked-in defaults preserve the exact replay-only experiment represented by the source notebook.

**Primary output:** `submission.zip`, containing the PEFT LoRA adapter files required by the competition.


## 1. Global configuration

The defaults below are configured for conservative continued training from the
existing 0.85 adapter. The adapter and OpenMathReasoning files are discovered
automatically from attached Kaggle inputs.


In [ ]:
# ============================================================
# Training hyperparameters
# ============================================================

LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192

# Conservative continued-training configuration.
NUM_STEPS = 400
BATCH_SIZE = 32
# Maximum number of sequences in one GPU micro-batch.
# Long sequences are automatically reduced to a singleton micro-batch.
MICRO_BATCH_SIZE = 2

# Bound padded activation volume. For example:
# - 2 sequences x 6,144 tokens
# - 1 sequence x 8,192 tokens
MAX_MICRO_BATCH_PADDED_TOKENS = 12_288
LEARNING_RATE = 1e-5

# Continue from the existing 0.85 adapter.
RESET_WEIGHTS = False

# LoRA targeting options. These must match the original adapter.
IN_PROJ_ONLY = False
MOE_TIE_WEIGHTS = True
ORIGINAL_PROBLEMS_ONLY = False
SHUFFLE_DATASET = False

# Runtime switches for optional diagnostic cells.
RUN_DATASET_STATS = False
RUN_DATASET_PREVIEW = False
RUN_SFT_SNAPSHOT_STATS = False
RUN_REPLAY_TOKENIZATION = True

# ============================================================
# Existing 0.85 adapter discovery
# ============================================================

# Use "AUTO" to search attached Kaggle notebook outputs.
# A direct submission.zip path can also be supplied.
PRETRAINED_ADAPTER_ZIP = "AUTO"

# Higher entries receive a larger discovery score.
PRETRAINED_ADAPTER_HINTS = (
    "nemotronlora01",
    "nemotronlora085",
    "nemotronlora0.85",
    "0.85",
)

# Temporary extraction location.
PRETRAINED_ADAPTER_EXTRACT_DIR = "/kaggle/tmp/pretrained_adapter_085"

# ============================================================
# Auxiliary-data mixture
# ============================================================

# OpenMath 15% produced no score gain, so it is disabled in this run.
USE_OPENMATH = False
RUN_OPENMATH_TOKENIZATION = False
OPENMATH_FINAL_RATIO = 0.0

# Replay Math is sampled to exactly 10% of the final example mixture.
REPLAY_FINAL_RATIO = 0.10
REPLAY_SEED = 42

# The generated OpenMathReasoning dataset contains at most 2,000 rows.
OPENMATH_MAX_ROWS = 2_000
OPENMATH_SEED = 42

# Use "AUTO" to search all attached Kaggle inputs.
OPENMATH_JSONL_PATH = "AUTO"
OPENMATH_TOKENIZED_PATH = "/kaggle/working/openmathreasoning_tokenized.jsonl"
OPENMATH_NOTEBOOK_HINTS = (
    "openmathreasoningcot2nemotron",
    "openmathreasoning",
)

assert RESET_WEIGHTS is False, "This notebook is intended for continued training."
assert 0.0 <= OPENMATH_FINAL_RATIO < 1.0
assert 0.0 < REPLAY_FINAL_RATIO < 1.0
assert REPLAY_FINAL_RATIO + OPENMATH_FINAL_RATIO < 1.0
assert OPENMATH_MAX_ROWS > 0
assert NUM_STEPS > 0
assert LEARNING_RATE > 0
assert MICRO_BATCH_SIZE >= 1
assert MAX_MICRO_BATCH_PADDED_TOKENS >= MAX_SEQ_LEN

# Modal upload target. Replace with your own Kaggle dataset handle when uploading from Modal.
KAGGLE_DATASET = "your-kaggle-username/nemotron-data"
MINUTES = 60

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
    "in_proj",
    "out_proj",
    "lm_head",
]

# Input data paths. Keep these paths unchanged unless your Kaggle inputs differ.
MATH_REPLAY_PATH = "/kaggle/input/datasets/mohamedamr992/replay-math/nemotron_math_1gb.jsonl"
MATH_TOKENIZED_PATH = "/kaggle/working/replay_math_tokenized.jsonl"

SFT_CORPUS_PATH = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14/tokens"
SFT_TRAIN_ORDER_PATH = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14/logprobs/index.jsonl"
COMPETITION_TRAIN_CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"

BASE_MODEL_HANDLE = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
MODAL_BASE_MODEL = "unsloth/Nemotron-3-Nano-30B-A3B"

# Offline package paths for Kaggle.
PACKAGE_WHEEL_DIR = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
CAUSAL_CONV1D_WHEEL = "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
MAMBA_SSM_WHEEL = "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
EXTRA_WHEEL_DIRS = ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]

# 8M answer tokens guarantees enough unique rows for a 10% mixture in the
# expected ~7.8K-example target SFT corpus, even for very long responses.
TARGET_REPLAY_ANSWER_TOKENS = 8_000_000

print("=" * 80)
print("CONTINUED-TRAINING CONFIGURATION")
print("=" * 80)
print(f"RESET_WEIGHTS            = {RESET_WEIGHTS}")
print(f"NUM_STEPS                = {NUM_STEPS}")
print(f"LEARNING_RATE             = {LEARNING_RATE:.2e}")
print(f"LOGICAL_BATCH_SIZE        = {BATCH_SIZE}")
print(f"MAX_MICRO_BATCH_SIZE      = {MICRO_BATCH_SIZE}")
print(f"MICRO_BATCH_TOKEN_BUDGET  = {MAX_MICRO_BATCH_PADDED_TOKENS:,}")
print(f"REPLAY_FINAL_RATIO        = {REPLAY_FINAL_RATIO:.1%}")
print(f"REPLAY_ANSWER_TOKEN_CAP   = {TARGET_REPLAY_ANSWER_TOKENS:,}")
print(f"USE_OPENMATH              = {USE_OPENMATH}")
print(f"OPENMATH_FINAL_RATIO      = {OPENMATH_FINAL_RATIO:.1%}")
print(f"OPENMATH_MAX_ROWS         = {OPENMATH_MAX_ROWS:,}")
print(f"PRETRAINED_ADAPTER_ZIP    = {PRETRAINED_ADAPTER_ZIP}")


## 2. Environment detection and offline installation

In [ ]:
import os

# Must be set before importing torch. This is secondary protection against
# allocator fragmentation; the main OOM fix is dynamic micro-batching.
os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)
import sys
import subprocess

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_MODAL_WORKER = "MODAL_TASK_ID" in os.environ
IS_MODAL_LAUNCHER = not IS_KAGGLE and not IS_MODAL_WORKER


def run_pip_install(args: list[str], check: bool = True) -> None:
    """Run pip with the current Python executable."""
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=check)


if IS_KAGGLE:
    run_pip_install([
        "-q",
        "--no-index",
        "--find-links",
        PACKAGE_WHEEL_DIR,
        "unsloth",
        "trl",
        "peft",
        "transformers",
        "datasets",
        "accelerate",
        "bitsandbytes",
    ])

    run_pip_install(["-q", CAUSAL_CONV1D_WHEEL])
    run_pip_install(["-q", MAMBA_SSM_WHEEL])

    for wheel_dir in EXTRA_WHEEL_DIRS:
        if os.path.isdir(wheel_dir):
            run_pip_install([
                "-q",
                "--no-index",
                "--find-links",
                wheel_dir,
                "protobuf==6.33.5",
                "sentencepiece",
                "safetensors",
                "huggingface_hub",
            ], check=False)

    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)

print(f"IS_KAGGLE={IS_KAGGLE}, IS_MODAL_WORKER={IS_MODAL_WORKER}, IS_MODAL_LAUNCHER={IS_MODAL_LAUNCHER}")
print("Environment setup complete.")

## 3. Input discovery and preflight checks

This section verifies both required notebook outputs before expensive model loading:

- Original 0.85 adapter `submission.zip`
- Filtered OpenMathReasoning `train_reasoning.jsonl`


In [ ]:
import json
from pathlib import Path


def count_jsonl_rows(jsonl_path: str) -> int:
    """Count rows in a JSONL file without loading it into memory."""
    row_count = 0
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for _ in f:
            row_count += 1
    return row_count


def preview_messages_rows(jsonl_path: str, n: int = 2, max_chars: int = 1200) -> None:
    """Print compact previews of message-format JSONL records."""
    shown = 0
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            print("=" * 100)
            print("ROW", shown + 1)
            print("KEYS:", list(row.keys()))
            print(json.dumps(row.get("messages", []), ensure_ascii=False, indent=2)[:max_chars])
            shown += 1
            if shown >= n:
                break


def _normalize_path_text(path: Path | str) -> str:
    return str(path).lower().replace("-", "").replace("_", "").replace(" ", "")


def resolve_pretrained_adapter_zip(path_setting: str) -> str:
    """Resolve the existing 0.85 adapter submission.zip from Kaggle inputs."""
    if path_setting and str(path_setting).upper() != "AUTO":
        direct_path = Path(path_setting)
        if not direct_path.is_file():
            raise FileNotFoundError(
                f"Configured PRETRAINED_ADAPTER_ZIP does not exist: {direct_path}"
            )
        print("Using configured pretrained adapter:")
        print(direct_path)
        return str(direct_path)

    search_root = Path("/kaggle/input") if IS_KAGGLE else Path("/data")
    print("\n" + "=" * 100)
    print("SEARCHING FOR THE EXISTING 0.85 ADAPTER")
    print("=" * 100)
    print("Search root:", search_root)

    candidates = sorted(search_root.rglob("submission.zip"))
    if not candidates:
        candidates = sorted(
            path for path in search_root.rglob("*.zip")
            if "adapter" in _normalize_path_text(path)
            or "nemotron" in _normalize_path_text(path)
        )

    normalized_hints = [
        hint.lower().replace("-", "").replace("_", "").replace(" ", "")
        for hint in PRETRAINED_ADAPTER_HINTS
    ]

    def score_candidate(path: Path) -> tuple[int, str]:
        normalized = _normalize_path_text(path)
        score = 0

        for hint_index, hint in enumerate(normalized_hints):
            if hint and hint in normalized:
                score += max(20, 200 - hint_index * 30)

        if path.name.lower() == "submission.zip":
            score += 20
        if "/notebooks/" in str(path).lower():
            score += 10
        if "nemotronlora01" in normalized:
            score += 300

        # Avoid accidentally choosing the OpenMath data notebook.
        if "openmathreasoningcot2nemotron" in normalized:
            score -= 500

        return score, str(path)

    ranked = sorted(candidates, key=score_candidate, reverse=True)

    if not ranked:
        raise FileNotFoundError(
            "No adapter ZIP was found. Add the original 0.85 notebook output "
            "(nemotronlora01) through Kaggle Add Input, then rerun this cell."
        )

    print(f"Found {len(ranked)} ZIP candidate(s):")
    for index, candidate in enumerate(ranked[:20], start=1):
        score, _ = score_candidate(candidate)
        print(f"  {index:2d}. score={score:4d}  {candidate}")

    selected = ranked[0]
    selected_score, _ = score_candidate(selected)

    if selected_score <= 20 and len(ranked) > 1:
        raise FileNotFoundError(
            "Multiple ZIP files were found, but none could be identified safely "
            "as the original 0.85 adapter. Set PRETRAINED_ADAPTER_ZIP to the "
            "exact submission.zip path printed above."
        )

    print("\nSelected pretrained 0.85 adapter:")
    print(selected)
    return str(selected)


def inspect_adapter_zip(zip_path: str) -> None:
    """Verify that the selected ZIP contains PEFT adapter files."""
    import zipfile

    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()

    config_files = [name for name in names if name.endswith("adapter_config.json")]
    weight_files = [
        name for name in names
        if name.endswith("adapter_model.safetensors")
        or name.endswith("adapter_model.bin")
    ]

    print("Adapter ZIP entries:", f"{len(names):,}")
    print("adapter_config.json:", config_files[:5])
    print("adapter weight file:", weight_files[:5])

    if not config_files:
        raise FileNotFoundError(
            f"The selected ZIP has no adapter_config.json: {zip_path}"
        )
    if not weight_files:
        raise FileNotFoundError(
            f"The selected ZIP has no adapter_model.safetensors/bin: {zip_path}"
        )


def resolve_openmath_jsonl(path_setting: str) -> str:
    """Resolve the OpenMathReasoning JSONL path from Kaggle notebook outputs."""
    if path_setting and str(path_setting).upper() != "AUTO":
        direct_path = Path(path_setting)
        if not direct_path.is_file():
            raise FileNotFoundError(
                f"Configured OPENMATH_JSONL_PATH does not exist: {direct_path}"
            )
        print("Using configured OpenMathReasoning path:")
        print(direct_path)
        return str(direct_path)

    search_root = Path("/kaggle/input") if IS_KAGGLE else Path("/data")
    print("\n" + "=" * 100)
    print("SEARCHING FOR OPENMATHREASONING DATA")
    print("=" * 100)
    print(f"Search root: {search_root}")

    candidates = list(search_root.rglob("train_reasoning.jsonl"))
    if not candidates:
        candidates = [
            path
            for path in search_root.rglob("*.jsonl")
            if "openmathreasoning" in _normalize_path_text(path)
        ]

    def score_candidate(path: Path) -> tuple[int, str]:
        normalized = _normalize_path_text(path)
        score = 0
        normalized_hints = [
            hint.lower().replace("-", "").replace("_", "")
            for hint in OPENMATH_NOTEBOOK_HINTS
        ]
        for hint_index, hint in enumerate(normalized_hints):
            if hint in normalized:
                score += 100 if hint_index == 0 else 50
        if "filtered2000" in normalized:
            score += 20
        if path.name == "train_reasoning.jsonl":
            score += 10
        if "/notebooks/" in str(path).lower():
            score += 5
        return score, str(path)

    ranked = sorted(candidates, key=score_candidate, reverse=True)
    if not ranked:
        raise FileNotFoundError(
            "OpenMathReasoning data was not found. Add the output of the "
            "OpenMathReasoningCoT2Nemotron notebook through Kaggle Add Input, "
            "then rerun this cell."
        )

    print(f"Found {len(ranked)} candidate JSONL file(s):")
    for index, candidate in enumerate(ranked[:10], start=1):
        score, _ = score_candidate(candidate)
        print(f"  {index:2d}. score={score:3d}  {candidate}")

    selected_path = ranked[0]
    selected_score, _ = score_candidate(selected_path)
    if selected_score < 50:
        raise FileNotFoundError(
            "JSONL files were found, but none looked like OpenMathReasoning output. "
            "Set OPENMATH_JSONL_PATH to the exact file path if necessary."
        )

    print("\nSelected OpenMathReasoning file:")
    print(selected_path)
    return str(selected_path)


# Resolve and validate the adapter before any expensive model work.
if not RESET_WEIGHTS:
    PRETRAINED_ADAPTER_ZIP = resolve_pretrained_adapter_zip(
        PRETRAINED_ADAPTER_ZIP
    )
    inspect_adapter_zip(PRETRAINED_ADAPTER_ZIP)
else:
    raise RuntimeError(
        "RESET_WEIGHTS=True is not allowed in this continued-training notebook."
    )

# Resolve OpenMathReasoning.
if USE_OPENMATH:
    OPENMATH_JSONL_PATH = resolve_openmath_jsonl(OPENMATH_JSONL_PATH)
    openmath_rows = count_jsonl_rows(OPENMATH_JSONL_PATH)
    print(f"OpenMathReasoning source rows: {openmath_rows:,}")
    if openmath_rows < OPENMATH_MAX_ROWS:
        print(
            f"WARNING: source has only {openmath_rows:,} rows, fewer than "
            f"OPENMATH_MAX_ROWS={OPENMATH_MAX_ROWS:,}."
        )
else:
    print("USE_OPENMATH=False: OpenMathReasoning auto-discovery skipped.")

if RUN_DATASET_STATS:
    print("Replay Math rows:", count_jsonl_rows(MATH_REPLAY_PATH))

if RUN_DATASET_PREVIEW:
    print("\nReplay Math preview")
    preview_messages_rows(MATH_REPLAY_PATH, n=2)
    if USE_OPENMATH:
        print("\nOpenMathReasoning preview")
        preview_messages_rows(OPENMATH_JSONL_PATH, n=2)

print("\n" + "=" * 100)
print("INPUT PREFLIGHT PASSED")
print("=" * 100)
print("0.85 adapter:", PRETRAINED_ADAPTER_ZIP)
print("OpenMath data:", OPENMATH_JSONL_PATH if USE_OPENMATH else "disabled")


In [ ]:
def estimate_sft_snapshot_stats(
    corpus_path: str,
    train_order_path: str,
    max_seq_len: int,
    batch_size: int,
) -> None:
    """Estimate example and token counts for the tokenized SFT snapshot."""
    ordered_ids = []
    seen = set()

    with open(train_order_path, "r") as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("epoch", 0) != 0:
                continue
            problem_id = rec["problem_id"]
            if problem_id in seen:
                continue
            seen.add(problem_id)
            ordered_ids.append(problem_id)

    valid_examples = 0
    total_tokens = 0
    total_unmasked_tokens = 0
    skipped_missing = 0
    skipped_empty = 0

    for problem_id in ordered_ids:
        segment_path = os.path.join(corpus_path, problem_id, "synthetic.json")
        if not os.path.isfile(segment_path):
            skipped_missing += 1
            continue

        with open(segment_path, "r") as f:
            rec = json.load(f)

        tokens = rec.get("tokens", [])[:max_seq_len]
        mask = rec.get("mask", [])[:max_seq_len]

        if len(tokens) < 2 or not any(mask):
            skipped_empty += 1
            continue

        valid_examples += 1
        total_tokens += len(tokens)
        total_unmasked_tokens += sum(mask)

    print("Ordered problem ids:", len(ordered_ids))
    print("Valid examples:", valid_examples)
    print("Skipped missing:", skipped_missing)
    print("Skipped empty/unmasked:", skipped_empty)
    print("Total tokens:", f"{total_tokens:,}")
    print("Total unmasked tokens:", f"{total_unmasked_tokens:,}")
    print("Full batches:", valid_examples // batch_size)


if RUN_SFT_SNAPSHOT_STATS:
    estimate_sft_snapshot_stats(
        corpus_path=SFT_CORPUS_PATH,
        train_order_path=SFT_TRAIN_ORDER_PATH,
        max_seq_len=MAX_SEQ_LEN,
        batch_size=BATCH_SIZE,
    )

## 4. Load tokenizer model for Replay Math


In [ ]:
RUN_ANY_AUX_TOKENIZATION = (
    RUN_REPLAY_TOKENIZATION
    or (USE_OPENMATH and RUN_OPENMATH_TOKENIZATION)
)

if RUN_ANY_AUX_TOKENIZATION:
    import gc
    import torch
    import kagglehub

    from unsloth import FastLanguageModel
    import causal_conv1d
    import mamba_ssm
    from causal_conv1d import causal_conv1d_fn

    print("Loading the base model once for auxiliary dataset tokenization...")
    MODEL_PATH = kagglehub.model_download(BASE_MODEL_HANDLE)
    print("MODEL_PATH:", MODEL_PATH)

    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0] * 10 + cc[1]}")
    print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
    print(f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )

    print("Model and tokenizer loaded successfully.")
    print("Tokenizer type:", type(tokenizer))
else:
    print(
        "Auxiliary tokenization is disabled. Ensure both requested tokenized "
        "files already exist before training."
    )


## 5. Inspect Replay Math chat-template rendering


In [ ]:
if RUN_ANY_AUX_TOKENIZATION:
    def inspect_chat_template(jsonl_path: str, label: str, max_chars: int = 2000) -> None:
        with open(jsonl_path, "r", encoding="utf-8") as f:
            row = json.loads(next(f))

        messages = row.get("messages", [])
        if len(messages) < 2:
            raise ValueError(f"{label} first row has fewer than two messages")

        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        assistant_message = messages[-1]
        reasoning = assistant_message.get("reasoning_content", "")
        final_content = assistant_message.get("content", "")

        print("=" * 100)
        print(label)
        print("Message count:", len(messages))
        print("Reasoning exists:", bool(reasoning))
        print("Final answer exists:", bool(final_content))
        print("Rendered text preview:")
        print(rendered[:max_chars])

    if RUN_REPLAY_TOKENIZATION:
        inspect_chat_template(MATH_REPLAY_PATH, "Replay Math template check")

    if USE_OPENMATH and RUN_OPENMATH_TOKENIZATION:
        inspect_chat_template(OPENMATH_JSONL_PATH, "OpenMathReasoning template check")


## 6. Tokenize Replay Math


In [ ]:
if RUN_ANY_AUX_TOKENIZATION:
    from itertools import islice
    from tqdm import tqdm

    def tokenize_message_jsonl(
        input_path: str,
        output_path: str,
        label: str,
        max_source_rows: int | None = None,
        target_answer_tokens: int | None = None,
    ) -> dict:
        """Tokenize message-format JSONL and save tokens plus assistant mask."""
        stats = {
            "source_rows": 0,
            "kept": 0,
            "skipped": 0,
            "truncated": 0,
            "total_tokens": 0,
            "answer_tokens": 0,
        }

        print("\n" + "=" * 100)
        print(f"TOKENIZING: {label}")
        print("Input:", input_path)
        print("Output:", output_path)
        if max_source_rows is not None:
            print(f"Maximum source rows: {max_source_rows:,}")
        if target_answer_tokens is not None:
            print(f"Target trainable answer tokens: {target_answer_tokens:,}")

        progress_total = max_source_rows

        with open(input_path, "r", encoding="utf-8") as fin, open(
            output_path, "w", encoding="utf-8"
        ) as fout:
            source_iterator = (
                fin
                if max_source_rows is None
                else islice(fin, max_source_rows)
            )
            progress = tqdm(
                source_iterator,
                total=progress_total,
                desc=label,
                unit="row",
                dynamic_ncols=True,
                mininterval=0.5,
            )

            for line in progress:
                stats["source_rows"] += 1

                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    stats["skipped"] += 1
                    continue

                messages = row.get("messages")
                if not messages or len(messages) < 2:
                    stats["skipped"] += 1
                    continue

                try:
                    prompt_ids = tokenizer.apply_chat_template(
                        messages[:-1],
                        tokenize=True,
                        add_generation_prompt=True,
                    )
                    full_ids = tokenizer.apply_chat_template(
                        messages,
                        tokenize=True,
                        add_generation_prompt=False,
                    )
                except Exception as exc:
                    stats["skipped"] += 1
                    if stats["skipped"] <= 3:
                        print(
                            f"\nWARNING: {label} row {stats['source_rows']} failed "
                            f"chat-template rendering: {type(exc).__name__}: {exc}"
                        )
                    continue

                if len(full_ids) <= len(prompt_ids):
                    stats["skipped"] += 1
                    continue

                if len(full_ids) > MAX_SEQ_LEN:
                    full_ids = full_ids[:MAX_SEQ_LEN]
                    stats["truncated"] += 1

                prompt_len = min(len(prompt_ids), len(full_ids))
                mask = [0] * prompt_len + [1] * (len(full_ids) - prompt_len)

                if len(full_ids) < 2 or not any(mask):
                    stats["skipped"] += 1
                    continue

                answer_tokens = sum(mask)
                fout.write(
                    json.dumps(
                        {
                            "tokens": full_ids,
                            "mask": mask,
                            "source": label,
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )

                stats["kept"] += 1
                stats["total_tokens"] += len(full_ids)
                stats["answer_tokens"] += answer_tokens

                progress.set_postfix(
                    kept=stats["kept"],
                    skipped=stats["skipped"],
                    truncated=stats["truncated"],
                    answer_M=f"{stats['answer_tokens'] / 1_000_000:.2f}",
                )

                if (
                    target_answer_tokens is not None
                    and stats["answer_tokens"] >= target_answer_tokens
                ):
                    break

            progress.close()

        print("-" * 100)
        print(f"{label} tokenization complete")
        print(f"Source rows read:       {stats['source_rows']:,}")
        print(f"Examples kept:          {stats['kept']:,}")
        print(f"Examples skipped:       {stats['skipped']:,}")
        print(f"Examples truncated:     {stats['truncated']:,}")
        print(f"Total tokens:           {stats['total_tokens']:,}")
        print(f"Trainable answer tokens:{stats['answer_tokens']:>13,}")
        print("Saved to:", output_path)

        if stats["kept"] == 0:
            raise RuntimeError(f"No valid examples were tokenized for {label}")

        return stats


    tokenization_summaries = {}

    if RUN_REPLAY_TOKENIZATION:
        tokenization_summaries["replay_math"] = tokenize_message_jsonl(
            input_path=MATH_REPLAY_PATH,
            output_path=MATH_TOKENIZED_PATH,
            label="Replay Math",
            max_source_rows=None,
            target_answer_tokens=TARGET_REPLAY_ANSWER_TOKENS,
        )
    else:
        print("RUN_REPLAY_TOKENIZATION=False: using existing", MATH_TOKENIZED_PATH)

    if USE_OPENMATH and RUN_OPENMATH_TOKENIZATION:
        tokenization_summaries["openmath"] = tokenize_message_jsonl(
            input_path=OPENMATH_JSONL_PATH,
            output_path=OPENMATH_TOKENIZED_PATH,
            label="OpenMathReasoning",
            max_source_rows=OPENMATH_MAX_ROWS,
            target_answer_tokens=None,
        )
    elif USE_OPENMATH:
        print(
            "RUN_OPENMATH_TOKENIZATION=False: using existing",
            OPENMATH_TOKENIZED_PATH,
        )

    print("\nAuxiliary tokenization summary:")
    print(json.dumps(tokenization_summaries, indent=2))


## 7. Continue training from the 0.85 adapter with 10% Replay Math


In [ ]:
def run_training() -> None:
    """Run the full LoRA training flow on Kaggle or inside a Modal worker."""
    import gc
    import json
    import math
    import random
    import subprocess
    import sys
    import time
    from collections import Counter

    from unsloth import FastLanguageModel

    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear

    # Environment-specific paths and optional adapter source
    if IS_KAGGLE:
        import kagglehub

        CORPUS_PATH = SFT_CORPUS_PATH
        TRAIN_ORDER_PATH = SFT_TRAIN_ORDER_PATH
        TRAIN_CSV_PATH = COMPETITION_TRAIN_CSV_PATH
        if RESET_WEIGHTS:
            raise RuntimeError(
                "This notebook requires RESET_WEIGHTS=False because it is "
                "configured to continue training the existing 0.85 adapter."
            )

        import shutil as _adapter_shutil
        import zipfile as _zipfile

        _adapter_zip = PRETRAINED_ADAPTER_ZIP
        _extract_root = PRETRAINED_ADAPTER_EXTRACT_DIR

        print("\n" + "=" * 100)
        print("EXTRACTING PRETRAINED 0.85 ADAPTER")
        print("=" * 100)
        print("Adapter ZIP:", _adapter_zip)
        print("Extraction root:", _extract_root)

        if os.path.isdir(_extract_root):
            _adapter_shutil.rmtree(_extract_root)
        os.makedirs(_extract_root, exist_ok=True)

        with _zipfile.ZipFile(_adapter_zip, "r") as _zf:
            _zf.extractall(_extract_root)

        # Support ZIP files with either root-level or nested adapter files.
        _adapter_dirs = []
        for _config_path in Path(_extract_root).rglob("adapter_config.json"):
            _candidate_dir = _config_path.parent
            if (
                (_candidate_dir / "adapter_model.safetensors").is_file()
                or (_candidate_dir / "adapter_model.bin").is_file()
            ):
                _adapter_dirs.append(_candidate_dir)

        if not _adapter_dirs:
            raise FileNotFoundError(
                "The extracted adapter ZIP does not contain a directory with both "
                "adapter_config.json and adapter_model.safetensors/bin."
            )

        _adapter_dirs = sorted(
            _adapter_dirs,
            key=lambda path: (len(path.parts), str(path)),
        )
        ADAPTER_SRC = str(_adapter_dirs[0])

        with open(os.path.join(ADAPTER_SRC, "adapter_config.json"), "r") as _f:
            _adapter_config = json.load(_f)

        print("Resolved adapter directory:", ADAPTER_SRC)
        print("Adapter rank:", _adapter_config.get("r"))
        print("Adapter alpha:", _adapter_config.get("lora_alpha"))
        print("Adapter target modules:", _adapter_config.get("target_modules"))

        _adapter_rank = _adapter_config.get("r")
        if _adapter_rank is not None:
            assert int(_adapter_rank) == LORA_RANK, (
                f"Adapter rank mismatch: ZIP has r={_adapter_rank}, "
                f"notebook uses LORA_RANK={LORA_RANK}."
            )

        MODEL_PATH = kagglehub.model_download(
            BASE_MODEL_HANDLE
        )
    else:  # IS_MODAL_WORKER
        MODEL_PATH = MODAL_BASE_MODEL
        CORPUS_PATH = "/data/corpus_preprocessed.jsonl"
        TRAIN_CSV_PATH = "/data/train.csv"
        ADAPTER_SRC = "/merged/weights"
        OUTPUT_DIR = "/output/weights"

    # GPU and CUDA extension sanity checks
    import causal_conv1d
    import mamba_ssm

    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0] * 10 + cc[1]}")
    print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
    print(
        f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}"
    )
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if IS_MODAL_WORKER:
        assert cc == (12, 0), (
            f"Expected sm_120 (RTX PRO 6000), got sm_{cc[0] * 10 + cc[1]}"
        )
    from causal_conv1d import causal_conv1d_fn

    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    # Clear stale HF modules cache (Modal-only; bug: persists across runs)
    if IS_MODAL_WORKER:
        import shutil as _shutil

        hf_modules = os.path.join(
            os.environ.get("HF_HOME", "/root/.cache/huggingface"), "modules"
        )
        if os.path.exists(hf_modules):
            _shutil.rmtree(hf_modules)

    # Load target SFT corpus into an examples list
    examples: list[dict] = []

    if IS_KAGGLE:
        # Load problem_ids in training order from logprobs/index.jsonl (epoch 0).
        # Each entry has {epoch, step, problem_id, ...}; epoch-0 entries are the
        # original training order, which we replay here.
        ordered_ids: list[str] = []
        seen: set[str] = set()
        with open(TRAIN_ORDER_PATH) as f:
            for line in f:
                rec = json.loads(line)
                if rec.get("epoch", 0) != 0:
                    continue
                pid = rec["problem_id"]
                if pid in seen:
                    continue
                seen.add(pid)
                ordered_ids.append(pid)
        print(
            f"Loaded {len(ordered_ids)} problem_ids in training order from "
            f"{TRAIN_ORDER_PATH}"
        )

        for sid in ordered_ids:
            seg_path = os.path.join(CORPUS_PATH, sid, "synthetic.json")
            assert os.path.isfile(seg_path), (
                f"problem_id {sid} from training order missing in corpus: {seg_path}"
            )
            with open(seg_path) as f:
                rec = json.load(f)
            tokens = rec["tokens"]
            mask = rec["mask"]
            if not tokens:
                continue
            if len(tokens) > MAX_SEQ_LEN:
                tokens = tokens[:MAX_SEQ_LEN]
                mask = mask[:MAX_SEQ_LEN]
            if not any(mask):
                continue
            examples.append(
                {
                    "problem_id": sid,
                    "tokens": tokens[:-1],
                    "targets": tokens[1:],
                    "weights": [float(m) for m in mask[1:]],
                    "source": "target_sft",
                }
            )
    else:  # IS_MODAL_WORKER
        with open(CORPUS_PATH) as f:
            for line in f:
                rec = json.loads(line.strip())
                tokens = rec["tokens"]
                mask = rec["mask"]
                if len(tokens) > MAX_SEQ_LEN:
                    tokens = tokens[:MAX_SEQ_LEN]
                    mask = mask[:MAX_SEQ_LEN]
                if not any(mask):
                    continue
                examples.append(
                    {
                        "problem_id": rec["problem_id"],
                        "tokens": tokens[:-1],
                        "targets": tokens[1:],
                        "weights": [float(m) for m in mask[1:]],
                        "source": "target_sft",
                    }
                )

    if ORIGINAL_PROBLEMS_ONLY:
        import csv

        with open(TRAIN_CSV_PATH) as f:
            original_ids = {row["id"] for row in csv.DictReader(f)}
        before = len(examples)
        examples = [e for e in examples if e["problem_id"] in original_ids]
        print(
            f"ORIGINAL_PROBLEMS_ONLY=True: filtered {before} → {len(examples)} examples "
            f"using {len(original_ids)} ids from {TRAIN_CSV_PATH}"
        )

    total_unmasked = sum(sum(e["weights"]) for e in examples)
    total_tokens = sum(len(e["tokens"]) for e in examples)
    print(
        f"Loaded {len(examples)} examples, {total_tokens:,} tokens "
        f"(unmasked={total_unmasked:,.0f})"
    )
    # ============================================================
    # Load and mix tokenized auxiliary examples
    # ============================================================

    def load_tokenized_examples(path: str, source_name: str) -> list[dict]:
        """Load tokenized JSONL examples in the training-loop format."""
        if not os.path.isfile(path):
            raise FileNotFoundError(
                f"Required tokenized dataset does not exist: {path}. "
                "Enable the corresponding tokenization switch and rerun the notebook."
            )

        loaded_examples: list[dict] = []
        skipped_rows = 0

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                tokens = rec.get("tokens", [])
                mask = rec.get("mask", [])

                if len(tokens) < 2 or len(tokens) != len(mask):
                    skipped_rows += 1
                    continue
                if not any(mask[1:]):
                    skipped_rows += 1
                    continue

                loaded_examples.append(
                    {
                        "tokens": tokens[:-1],
                        "targets": tokens[1:],
                        "weights": [float(m) for m in mask[1:]],
                        "source": source_name,
                    }
                )

        print(
            f"Loaded {len(loaded_examples):,} {source_name} examples "
            f"from {path}; skipped={skipped_rows:,}"
        )
        return loaded_examples


    def interleave_evenly(base: list[dict], auxiliary: list[dict]) -> list[dict]:
        """Insert auxiliary examples approximately uniformly through base examples."""
        if not auxiliary:
            return list(base)
        if not base:
            return list(auxiliary)

        mixed: list[dict] = []
        aux_index = 0
        insertion_interval = len(base) / len(auxiliary)
        next_insertion = insertion_interval

        for base_index, example in enumerate(base, start=1):
            mixed.append(example)
            while aux_index < len(auxiliary) and base_index >= next_insertion:
                mixed.append(auxiliary[aux_index])
                aux_index += 1
                next_insertion += insertion_interval

        mixed.extend(auxiliary[aux_index:])
        return mixed


    print("\n" + "=" * 100)
    print("AUXILIARY DATA MIXING")
    print("=" * 100)
    print("Original target SFT examples:", f"{len(examples):,}")

    all_replay_examples = load_tokenized_examples(
        MATH_TOKENIZED_PATH,
        "replay_math",
    )
    assert all_replay_examples, "No Replay Math examples were loaded"

    # Let N be the original target-SFT count and R the selected replay count.
    # R / (N + R) = REPLAY_FINAL_RATIO.
    desired_replay_count = round(
        len(examples) * REPLAY_FINAL_RATIO / (1.0 - REPLAY_FINAL_RATIO)
    )
    replay_count = min(desired_replay_count, len(all_replay_examples))

    if replay_count < desired_replay_count:
        print(
            "WARNING: not enough unique Replay Math rows to reach the requested "
            f"{REPLAY_FINAL_RATIO:.1%} final ratio. Requested "
            f"{desired_replay_count:,}, available {len(all_replay_examples):,}."
        )

    if replay_count < len(all_replay_examples):
        replay_rng = random.Random(REPLAY_SEED)
        replay_indices = sorted(
            replay_rng.sample(range(len(all_replay_examples)), replay_count)
        )
        replay_examples = [all_replay_examples[i] for i in replay_indices]
    else:
        replay_examples = all_replay_examples

    examples = interleave_evenly(examples, replay_examples)
    actual_replay_ratio = len(replay_examples) / len(examples)

    print("Replay Math available:", f"{len(all_replay_examples):,}")
    print("Replay Math selected:", f"{len(replay_examples):,}")
    print("Requested Replay Math final ratio:", f"{REPLAY_FINAL_RATIO:.2%}")
    print("Actual Replay Math final ratio:", f"{actual_replay_ratio:.2%}")
    print("Examples after Replay Math:", f"{len(examples):,}")

    if USE_OPENMATH:
        all_openmath_examples = load_tokenized_examples(
            OPENMATH_TOKENIZED_PATH,
            "openmath",
        )
        assert all_openmath_examples, "No OpenMathReasoning examples were loaded"

        desired_openmath_count = round(
            len(examples) * OPENMATH_FINAL_RATIO / (1.0 - OPENMATH_FINAL_RATIO)
        )
        openmath_count = min(desired_openmath_count, len(all_openmath_examples))

        if openmath_count < desired_openmath_count:
            print(
                "WARNING: not enough OpenMathReasoning rows to reach the requested "
                f"{OPENMATH_FINAL_RATIO:.1%} final ratio. Requested "
                f"{desired_openmath_count:,}, available {len(all_openmath_examples):,}."
            )

        if openmath_count < len(all_openmath_examples):
            rng = random.Random(OPENMATH_SEED)
            selected_indices = sorted(
                rng.sample(range(len(all_openmath_examples)), openmath_count)
            )
            openmath_examples = [all_openmath_examples[i] for i in selected_indices]
        else:
            openmath_examples = all_openmath_examples

        examples = interleave_evenly(examples, openmath_examples)
        actual_openmath_ratio = len(openmath_examples) / len(examples)

        print("OpenMathReasoning available:", f"{len(all_openmath_examples):,}")
        print("OpenMathReasoning selected:", f"{len(openmath_examples):,}")
        print("Requested final ratio:", f"{OPENMATH_FINAL_RATIO:.2%}")
        print("Actual final ratio:", f"{actual_openmath_ratio:.2%}")
    else:
        print("USE_OPENMATH=False: no OpenMathReasoning examples added.")

    source_counts = Counter(example.get("source", "unknown") for example in examples)
    print("\nFinal mixed-example counts:")
    for source_name, count in source_counts.items():
        print(f"  {source_name:16s}: {count:>8,} ({count / len(examples):6.2%})")
    print(f"  {'TOTAL':16s}: {len(examples):>8,}")
    # Load base model
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if IS_MODAL_WORKER:
        hf_cache_vol.commit()  # noqa: F821 — defined at module level on non-Kaggle

    # Attach LoRA adapters
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
    )
    FastLanguageModel.for_training(model)

    # ── Patch Mamba CUDA fast path ───────────────────────────────────
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m
            break
    assert nemotron_mod is not None, "Could not find modeling_nemotron_h module"
    print(f"is_fast_path_available was: {nemotron_mod.is_fast_path_available}")
    nemotron_mod.is_fast_path_available = True  # type: ignore[attr-defined]
    print("Patched is_fast_path_available = True")

    # ── Manually add lm_head LoRA (Unsloth drops it for MoE) ─────────
    _causal_lm = model
    while hasattr(_causal_lm, "model"):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            _cfg,
            "default",
            target=_lm_head,
            target_name="lm_head",
            parent=_causal_lm,
        )
        print("Manually added LoRA to lm_head")
    else:
        print("lm_head already has LoRA")

    # ── Cast LoRA params to fp32 (base model stays bf16 except MoE router) ──
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)

    for name, param in model.named_parameters():
        if ".lora_" in name:
            assert param.dtype == torch.float32, (
                f"LoRA param {name} expected fp32, got {param.dtype}"
            )
            continue

        is_router = (
            ".mixer.gate." in name
        )  # NemotronHTopkRouter.weight + e_score_correction_bias
        # Nemotron-H loads the MoE router (`mixer.gate`) in fp32 on purpose.
        # Ref: transformers/src/transformers/models/nemotron_h/modeling_nemotron_h.py
        #
        #   class NemotronHPreTrainedModel(PreTrainedModel):
        #       _keep_in_fp32_modules_strict = ["e_score_correction_bias"]
        #
        #   class NemotronHTopkRouter(nn.Module):
        #       def __init__(self, config):
        #           self.weight = nn.Parameter(torch.empty((self.n_routed_experts, config.hidden_size)))
        #           self.register_buffer("e_score_correction_bias", torch.zeros(self.n_routed_experts))
        #       def forward(self, hidden_states):
        #           router_logits = F.linear(
        #               hidden_states.type(torch.float32),
        #               self.weight.type(torch.float32),
        #           )
        #           return router_logits
        #
        # The per-forward fp32 cast on `self.weight` plus the strict list entry
        # mean the gate weight is promoted to fp32 at load time.
        if is_router:
            assert param.dtype == torch.float32, (
                f"param {name} expected fp32, got {param.dtype}"
            )
            continue

        assert param.dtype == torch.bfloat16, (
            f"param {name} expected bf16, got {param.dtype}"
        )
        continue

    print("Verified: LoRA params fp32, base params bf16 (MoE router fp32)")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model: {trainable:,} trainable / {total:,} total parameters")

    # ── Patch forward with Cut Cross-Entropy ─────────────────────────
    _base = model
    while hasattr(_base, "model"):
        _base = _base.model

    def _patched_causal_forward(
        input_ids=None, attention_mask=None, labels=None, **kwargs
    ):
        backbone_out = _base.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **{
                k: v
                for k, v in kwargs.items()
                if k in ("position_ids", "past_key_values", "use_cache")
            },
        )
        hidden_states = backbone_out[0]
        lm_head = _base.lm_head
        base_w = lm_head.base_layer.weight
        lora_A = lm_head.lora_A["default"].weight
        lora_B = lm_head.lora_B["default"].weight
        scaling = lm_head.scaling["default"]
        lm_weight = base_w + scaling * lora_B @ lora_A
        if labels is not None:
            per_token_ce = linear_cross_entropy(
                hidden_states, lm_weight, labels, reduction="none"
            )
            loss = per_token_ce.mean()
        else:
            per_token_ce = None
            loss = None
        model._cached_per_token_ce = per_token_ce  # type: ignore[attr-defined]
        return loss

    _base.forward = _patched_causal_forward
    print("Patched CausalLM.forward with CCE (no logits materialization)")

    # ── Load adapter weights (unless RESET_WEIGHTS) ──────────────────
    if RESET_WEIGHTS:
        print(
            "RESET_WEIGHTS=True — skipping pretrained adapter load; using fresh LoRA init"
        )
        loaded = 0
        adapter_weights: dict = {}
    else:
        print(f"Loading adapter from {ADAPTER_SRC}...")
        from peft import load_peft_weights

        adapter_weights = load_peft_weights(ADAPTER_SRC)
        if not adapter_weights:
            raise RuntimeError(
                f"No PEFT weights were loaded from {ADAPTER_SRC}."
            )
        print(f"Adapter tensors found: {len(adapter_weights):,}")

        model_sd = model.state_dict()
        new_sd: dict = {}
        loaded = 0
        for ak, av in adapter_weights.items():
            if ak in model_sd:
                new_sd[ak] = av
                loaded += 1
                continue
            ak_with_default = ak.replace(
                ".lora_A.weight", ".lora_A.default.weight"
            ).replace(".lora_B.weight", ".lora_B.default.weight")
            if ak_with_default in model_sd:
                new_sd[ak_with_default] = av
                loaded += 1
                continue
            ak_lm = ak.replace(".backbone.lm_head.", ".lm_head.")
            ak_lm_default = ak_lm.replace(
                ".lora_A.weight", ".lora_A.default.weight"
            ).replace(".lora_B.weight", ".lora_B.default.weight")
            if ak_lm_default in model_sd:
                new_sd[ak_lm_default] = av
                loaded += 1
                continue

        model.load_state_dict(new_sd, strict=False)
        assert loaded == len(adapter_weights), (
            f"Not all adapter weights loaded: {loaded}/{len(adapter_weights)}"
        )
        print(f"  Loaded {loaded}/{len(adapter_weights)} weights into model")
        print("Confirmed: training will continue from the existing 0.85 adapter.")

    # ── Freeze all LoRA params except in_proj (if IN_PROJ_ONLY) ──
    print(f"{IN_PROJ_ONLY=}")
    if IN_PROJ_ONLY:
        for name, param in model.named_parameters():
            if param.requires_grad and ".in_proj." not in name:
                param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  {trainable_params:,} trainable / {frozen_params:,} frozen")

    # MoE expert LoRA weight tying
    # Tie the LoRA side that touches the hidden dimension:
    #   gate_up_proj / up_proj / w1 / gate_proj  -> tie A (input/hidden side)
    #   down_proj / w2                           -> tie B (output/hidden side)
    # We keep Unsloth's batched [num_experts, ...] tensor layout; "tying" means
    # all 128 expert slices are kept identical. Saving the adapter naturally
    # emits 128 per-expert copies, so submission.zip is untied downstream.
    moe_tied_params: list[torch.Tensor] = []
    if MOE_TIE_WEIGHTS:
        w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_proj_names = ("down_proj", ".w2.")
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if ".experts." not in name or ".lora_" not in name:
                continue
            is_w1 = any(p in name for p in w1_proj_names)
            is_w2 = any(p in name for p in w2_proj_names)
            is_A = ".lora_A." in name
            is_B = ".lora_B." in name
            should_tie = (is_w1 and is_A) or (is_w2 and is_B)
            if not should_tie:
                continue
            if param.dim() < 2 or param.shape[0] <= 1:
                continue
            moe_tied_params.append(param)

        def _tie_param_init() -> None:
            """Make all 128 expert slices identical (mean-and-broadcast)."""
            with torch.no_grad():
                for p in moe_tied_params:
                    mean = p.data.mean(dim=0, keepdim=True)
                    p.data.copy_(mean.expand_as(p.data))

        def _tie_grads() -> None:
            # Sum (not mean) across the expert dim: if W is the shared LoRA factor
            # and each expert uses a copy W_i = W, chain rule gives
            # dL/dW = sum_i dL/dW_i. Inactive experts contribute 0 and router
            # weights are already baked into active g_i, so there's no
            # double-counting. Summing keeps all 128 slices identical after each
            # AdamW step and reproduces the true shared-weight update; mean would
            # be off by a 1/128 lr rescale (and not exactly equivalent under
            # AdamW's eps/weight-decay).
            with torch.no_grad():
                for p in moe_tied_params:
                    if p.grad is None:
                        continue
                    grad_sum = p.grad.sum(dim=0, keepdim=True)
                    p.grad.copy_(grad_sum.expand_as(p.grad))

        print(f"MoE weight tying: {len(moe_tied_params)} params identified for tying")
        if moe_tied_params:
            print(f"  example shapes: {[tuple(p.shape) for p in moe_tied_params[:3]]}")
        _tie_param_init()  # start from a tied state
    else:

        def _tie_grads() -> None:
            pass

    # Training loop
    gc.collect()
    torch.cuda.empty_cache()

    device = next(model.parameters()).device
    optimizer: torch.optim.AdamW | None = None

    indices = list(range(len(examples)))
    if SHUFFLE_DATASET:
        rng = random.Random(0)
        rng.shuffle(indices)
        print(f"SHUFFLE_DATASET=True: shuffled {len(indices)} examples (seed=0)")
    else:
        print(f"SHUFFLE_DATASET=False: keeping corpus order ({len(indices)} examples)")

    training_log: list[str] = []

    def _log(msg: str) -> None:
        print(msg, flush=True)
        training_log.append(msg)

    max_steps = len(examples) // BATCH_SIZE
    num_steps = NUM_STEPS
    if num_steps > max_steps:
        _log(
            f"WARNING: NUM_STEPS={NUM_STEPS} exceeds max_steps={max_steps} "
            f"({len(examples)} examples // {BATCH_SIZE} batch). Clamping to {max_steps}."
        )
        num_steps = max_steps

    _log(
        f"Training: {num_steps} steps, logical_batch_size={BATCH_SIZE}, "
        f"max_micro_batch_size={MICRO_BATCH_SIZE}, "
        f"micro_batch_token_budget={MAX_MICRO_BATCH_PADDED_TOKENS}, "
        f"lr={LEARNING_RATE}"
    )

    step = 0
    for batch_start in range(0, len(indices), BATCH_SIZE):
        if step >= num_steps:
            break
        batch_indices = indices[batch_start : batch_start + BATCH_SIZE]
        batch = [examples[i] for i in batch_indices]
        batch_tokens = [e["tokens"] for e in batch]
        batch_targets = [e["targets"] for e in batch]
        batch_weights = [e["weights"] for e in batch]

        n = len(batch)

        # ------------------------------------------------------------
        # Build OOM-safe, length-aware micro-batches.
        #
        # Padding cost is approximately:
        #     number_of_sequences * longest_sequence_in_micro_batch
        #
        # Sorting only inside the current logical batch preserves the
        # dataset composition while greatly reducing wasted padding.
        # ------------------------------------------------------------
        length_sorted_positions = sorted(
            range(n),
            key=lambda position: len(batch_tokens[position]),
        )

        micro_batch_positions = []
        current_positions = []
        current_max_len = 0

        for position in length_sorted_positions:
            sequence_len = len(batch_tokens[position])
            proposed_count = len(current_positions) + 1
            proposed_max_len = max(current_max_len, sequence_len)
            proposed_padded_tokens = proposed_count * proposed_max_len

            exceeds_sequence_cap = proposed_count > MICRO_BATCH_SIZE
            exceeds_token_budget = (
                proposed_padded_tokens > MAX_MICRO_BATCH_PADDED_TOKENS
            )

            if current_positions and (
                exceeds_sequence_cap or exceeds_token_budget
            ):
                micro_batch_positions.append(current_positions)
                current_positions = [position]
                current_max_len = sequence_len
            else:
                current_positions.append(position)
                current_max_len = proposed_max_len

        if current_positions:
            micro_batch_positions.append(current_positions)

        n_accum = len(micro_batch_positions)
        total_loss_sum = 0.0
        total_weight_sum = 0.0

        # Exact normalization across variable-size micro-batches.
        logical_batch_weight_sum = sum(
            float(sum(weights))
            for weights in batch_weights
        )
        if logical_batch_weight_sum <= 0:
            logical_batch_weight_sum = 1.0

        micro_batch_plan = [
            {
                "sequences": len(positions),
                "max_len": max(len(batch_tokens[p]) for p in positions),
                "padded_tokens": (
                    len(positions)
                    * max(len(batch_tokens[p]) for p in positions)
                ),
            }
            for positions in micro_batch_positions
        ]

        print(
            f"  logical batch {step + 1}: {n} sequences -> "
            f"{n_accum} micro-batches; "
            f"largest padded micro-batch="
            f"{max(plan['padded_tokens'] for plan in micro_batch_plan):,} tokens"
        )

        for mb_idx, positions in enumerate(micro_batch_positions):
            mb_toks = [batch_tokens[position] for position in positions]
            mb_tgts = [batch_targets[position] for position in positions]
            mb_wts = [batch_weights[position] for position in positions]

            n_micro = len(mb_toks)
            max_len = max(len(tokens) for tokens in mb_toks)
            total_len = sum(len(tokens) for tokens in mb_toks)
            padded_token_count = n_micro * max_len

            padded_input = torch.zeros(
                n_micro, max_len, dtype=torch.long, device=device
            )
            padded_targets = torch.zeros(
                n_micro, max_len, dtype=torch.long, device=device
            )
            padded_weights = torch.zeros(
                n_micro, max_len, dtype=torch.float32, device=device
            )
            attention_mask = torch.zeros(
                n_micro, max_len, dtype=torch.long, device=device
            )

            for local_index in range(n_micro):
                seq_len = len(mb_toks[local_index])
                padded_input[local_index, :seq_len] = torch.as_tensor(
                    mb_toks[local_index],
                    dtype=torch.long,
                    device=device,
                )
                padded_targets[local_index, :seq_len] = torch.as_tensor(
                    mb_tgts[local_index],
                    dtype=torch.long,
                    device=device,
                )
                padded_weights[local_index, :seq_len] = torch.as_tensor(
                    mb_wts[local_index],
                    dtype=torch.float32,
                    device=device,
                )
                attention_mask[local_index, :seq_len] = 1

            torch.cuda.reset_peak_memory_stats()
            t0 = time.time()

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                model(
                    input_ids=padded_input,
                    attention_mask=attention_mask,
                    labels=padded_targets,
                    use_cache=False,
                )
                per_token_ce = model._cached_per_token_ce  # type: ignore[attr-defined]
                weighted_loss = per_token_ce * padded_weights
                weight_sum_t = padded_weights.sum()
                loss_sum_t = weighted_loss.sum()

                # Normalize every micro-batch by the total supervised-token
                # weight of the entire logical batch. This makes gradient
                # accumulation exact even when micro-batch sizes differ.
                loss = loss_sum_t / logical_batch_weight_sum

            loss.backward()

            total_loss_sum += float(loss_sum_t.detach().item())
            total_weight_sum += float(weight_sum_t.detach().item())

            t_end = time.time()
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            mem_gb = torch.cuda.memory_allocated() / 1e9

            print(
                f"    micro-batch {mb_idx + 1}/{n_accum}: "
                f"{n_micro} seqs, max_len={max_len}, "
                f"actual_tokens={total_len:,}, "
                f"padded_tokens={padded_token_count:,}, "
                f"wall={t_end - t0:.1f}s, "
                f"peak={peak_gb:.1f}GB, mem={mem_gb:.1f}GB"
            )

            # Critical cleanup:
            # the patched forward stores per_token_ce on the model object.
            # Clear that reference so the completed autograd graph can be
            # released before the next micro-batch.
            model._cached_per_token_ce = None  # type: ignore[attr-defined]

            del (
                loss,
                per_token_ce,
                weighted_loss,
                loss_sum_t,
                weight_sum_t,
                padded_input,
                padded_targets,
                padded_weights,
                attention_mask,
            )

        if optimizer is None:
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=LEARNING_RATE,
                betas=(0.9, 0.95),
                eps=1e-8,
                weight_decay=0.0,
            )
        lr = LEARNING_RATE * (1 - step / num_steps)
        for pg in optimizer.param_groups:
            pg["lr"] = lr
        _tie_grads()  # sum tied MoE expert gradients before clip and optimizer step
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1e9
        )
        optimizer.step()
        optimizer.zero_grad()
        loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
        step += 1
        _log(
            f"  step {step}/{num_steps}: "
            f"loss:mean={loss_mean:.6f}, grad_norm={grad_norm:.4f}, lr={lr:.2e}"
        )

    print(
        f"\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
    )

    # ── Save adapter + rename lm_head keys (identical on both sides) ──
    from safetensors.torch import load_file, save_file

    save_dir = "." if IS_KAGGLE else OUTPUT_DIR
    os.makedirs(save_dir, exist_ok=True)
    for _f in os.listdir(save_dir):
        if _f.startswith("adapter"):
            os.remove(os.path.join(save_dir, _f))
    model.save_pretrained(save_dir)
    st_path = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st_path)
    renamed = {
        k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
        for k, v in tensors.items()
    }
    save_file(renamed, st_path)

    # ── Clean unsloth compiled cache (runs on both) ──────────────────
    _ucache = "unsloth_compiled_cache"
    if os.path.isdir(_ucache):
        import shutil as _sh

        _sh.rmtree(_ucache)

    # Package output artifacts
    if IS_KAGGLE:
        import zipfile

        adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
        SUBMISSION_ZIP = "submission.zip"
        with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in adapter_files:
                zf.write(os.path.join(save_dir, fname), fname)
        for fname in adapter_files:
            os.remove(os.path.join(save_dir, fname))
        print(f"Wrote {SUBMISSION_ZIP}")
    else:  # IS_MODAL_WORKER
        import shutil
        import tempfile

        with open(os.path.join(save_dir, "training_log.txt"), "w") as f:
            f.write("\n".join(training_log) + "\n")
        output_vol.commit()  # noqa: F821 — defined at module level on non-Kaggle

        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(os.path.join(kaggle_dir, "access_token"), "w") as f:
            f.write(os.environ["KAGGLE_API_TOKEN"])
        upload_dir = tempfile.mkdtemp()
        for fname in os.listdir(save_dir):
            shutil.copy(os.path.join(save_dir, fname), upload_dir)
        metadata = {"id": KAGGLE_DATASET, "title": KAGGLE_DATASET.split("/")[1]}
        with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as f:
            json.dump(metadata, f)
        print(f"Uploading to Kaggle {KAGGLE_DATASET}...")
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                upload_dir,
                "-m",
                "post-finetuned adapter + compiled wheels",
            ],
            check=True,
        )
        print("Kaggle upload complete.")
    print("Training complete.")

## 8. Optional Modal launcher

In [ ]:
# Modal entrypoint and worker image. This block is skipped on Kaggle.
if not IS_KAGGLE:
    import modal

    train_image = (
        modal.Image.from_registry(
            "nvidia/cuda:12.8.1-devel-ubuntu22.04",
            add_python="3.12",
        )
        .entrypoint([])
        .apt_install("git", "build-essential", "clang")
        .pip_install(
            "torch==2.10.0",
            extra_index_url="https://download.pytorch.org/whl/cu128",
        )
        .pip_install(
            "safetensors>=0.5.0",
            "transformers>=4.56.2",
            "accelerate>=1.0.0",
            "peft>=0.15.0",
            "bitsandbytes>=0.45.0",
            "huggingface_hub>=0.36.2",
            "hf-transfer>=0.1.9",
            "numpy",
            "pillow",
            "torchvision",
            "datasets",
            "sentencepiece",
            "xformers",
            "cut-cross-entropy>=25.1.0",
            "wheel",
            "setuptools",
            "trl",
            "kaggle>=1.6.0",
        )
        .run_commands(
            'python -c "import torch.utils.cpp_extension as e; p=e.__file__; '
            "t=open(p).read().replace('raise RuntimeError(CUDA_MISMATCH_MESSAGE', 'pass  # '); "
            "open(p,'w').write(t)\"",
            "TORCH_CUDA_ARCH_LIST='12.0' pip wheel --no-build-isolation --wheel-dir /wheels mamba_ssm==2.3.1 causal_conv1d==1.6.1",
            "pip install --no-deps /wheels/mamba_ssm-*.whl /wheels/causal_conv1d-*.whl",
            "pip install --no-deps 'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo'",
            "pip install --no-deps 'unsloth[base] @ git+https://github.com/unslothai/unsloth'",
        )
        .pip_install("einops")
        .env({"HF_HOME": "/root/.cache/huggingface"})
    )

    hf_cache_vol = modal.Volume.from_name("huggingface-cache", create_if_missing=True)
    merged_vol = modal.Volume.from_name("merged-adapter", create_if_missing=True)
    corpus_vol = modal.Volume.from_name("corpus-data", create_if_missing=True)
    output_vol = modal.Volume.from_name("post-finetune-output", create_if_missing=True)

    app = modal.App("post-finetune-pipeline")

    @app.function(
        image=train_image,
        gpu="RTX-PRO-6000",
        volumes={
            "/root/.cache/huggingface": hf_cache_vol,
            "/merged": merged_vol,
            "/data": corpus_vol,
            "/output": output_vol,
        },
        timeout=6 * 60 * MINUTES,
        secrets=[modal.Secret.from_local_environ(["KAGGLE_API_TOKEN"])],
    )
    def train_remote() -> None:
        run_training()

    if IS_MODAL_LAUNCHER:

        @app.local_entrypoint()
        def main() -> None:
            train_remote.remote()

## 9. Start Replay Math continued training


In [ ]:
# Clear the temporary tokenization model before loading the training model.
import gc
import torch

if "model" in globals():
    del model
if "tokenizer" in globals():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()
print("Temporary Replay Math tokenization objects cleared from GPU.")

print("\n" + "=" * 100)
print("STARTING CONTINUED TRAINING")
print("=" * 100)
print("Pretrained adapter:", PRETRAINED_ADAPTER_ZIP)
print("Training steps:", NUM_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Replay Math target ratio:", f"{REPLAY_FINAL_RATIO:.1%}")
print("OpenMath enabled:", USE_OPENMATH)

# On Kaggle, run training directly. On Modal worker, train_remote() calls run_training().
if IS_KAGGLE:
    run_training()
